# NVIDIA Stock Price Time-Series Forecasting Engine
### Augmented Dickey-Fuller (ADF) Tests | Box-Jenkins ARIMA(2,1,1) | Holt Exponential Smoothing | Out-of-Sample Backtesting

This econometric and time-series engineering pipeline demonstrates:
1. **Stationarity Diagnostics:** Executing Augmented Dickey-Fuller (ADF) unit-root tests to diagnose non-stationarity in raw prices ($t = 0.97$) and establish stationarity after first differencing ($t = -37.84, p < 0.001$).
2. **Box-Jenkins ARIMA(2,1,1) Formulation:** Fitting autoregressive integrated moving average processes on log returns.
3. **60-Day Forward Backtesting:** Generating multi-step out-of-sample forward forecasts across 60 trading days.
4. **Error Metrics:** Evaluating performance against Holt Linear Exponential Smoothing with **10.19% MAPE**.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Add root directory to path
sys.path.insert(0, os.getcwd())

from src.data_loader import NVDADataLoader
from src.time_series_forecaster import NVDATimeSeriesForecaster

# 1. Ingest 5-Year Historical Daily Trading Prices
loader = NVDADataLoader(data_dir="data")
data = loader.load_data(test_days=60)

print(f"Total Historical Trading Days : {len(data['full_df']):,}")
print(f"In-Sample Training Horizon    : {len(data['train_df']):,} days")
print(f"Out-of-Sample Test Window     : {len(data['test_df']):,} days")
print(f"Historical Price Range        : ${data['full_df']['Close'].min():.2f} -> ${data['full_df']['Close'].max():.2f}")

Total Historical Trading Days : 1,390
In-Sample Training Horizon    : 1,330 days
Out-of-Sample Test Window     : 60 days
Historical Price Range        : $25.78 -> $1051.58


## 2. Augmented Dickey-Fuller (ADF) Unit Root Stationarity Tests

In [3]:
forecaster = NVDATimeSeriesForecaster(data)

adf_raw = forecaster.run_adf_stationarity_test(series=data['full_df']['Close'].values)
adf_diff = forecaster.run_adf_stationarity_test(series=np.diff(np.log(data['full_df']['Close'].values)))

print("=" * 85)
print("AUGMENTED DICKEY-FULLER (ADF) STATIONARITY TEST RESULTS")
print("=" * 85)
print(f"• Raw Price Series ADF Stat        : t = {adf_raw['adf_t_statistic']:>8.4f} (5% Critical: {adf_raw['critical_value_5pct']}) -> {adf_raw['conclusion']}")
print(f"• First-Differenced Series ADF Stat : t = {adf_diff['adf_t_statistic']:>8.4f} (5% Critical: {adf_diff['critical_value_5pct']}) -> {adf_diff['conclusion']}")
print("=" * 85)

AUGMENTED DICKEY-FULLER (ADF) STATIONARITY TEST RESULTS
• Raw Price Series ADF Stat        : t =   1.0816 (5% Critical: -2.86) -> Non-Stationary (Contains Unit Root -> Differencing d=1 required)
• First-Differenced Series ADF Stat : t = -37.8420 (5% Critical: -2.86) -> Stationary (Reject Unit Root)


## 3. Box-Jenkins ARIMA(2,1,1) & Holt Smoothing Out-of-Sample Backtesting

In [5]:
# Fit and forecast ARIMA(2,1,1)
arima_fc = forecaster.fit_and_forecast_arima(p=2, d=1, q=1, forecast_horizon=60)
arima_metrics = forecaster.evaluate_forecast(arima_fc)

# Fit and forecast Holt's Linear Smoothing
holt_fc = forecaster.fit_and_forecast_holt(forecast_horizon=60)
holt_metrics = forecaster.evaluate_forecast(holt_fc)

res_df = pd.DataFrame([
    {'Model': 'ARIMA (2,1,1) Multi-Step', 'RMSE ($)': f"${arima_metrics['rmse']:.2f}", 'MAE ($)': f"${arima_metrics['mae']:.2f}", 'MAPE (%)': f"{arima_metrics['mape_pct']:.2f}%", 'Directional Acc': f"{arima_metrics['directional_accuracy_pct']:.2f}%"},
    {'Model': 'Holt Linear Smoothing', 'RMSE ($)': f"${holt_metrics['rmse']:.2f}", 'MAE ($)': f"${holt_metrics['mae']:.2f}", 'MAPE (%)': f"{holt_metrics['mape_pct']:.2f}%", 'Directional Acc': f"{holt_metrics['directional_accuracy_pct']:.2f}%"}
])

print("=" * 85)
print("OUT-OF-SAMPLE FORECASTING PERFORMANCE BENCHMARK (60-DAY TEST HORIZON)")
print("=" * 85)
print(res_df.to_string(index=False))
print("=" * 85)

OUT-OF-SAMPLE FORECASTING PERFORMANCE BENCHMARK (60-DAY TEST HORIZON)
                   Model RMSE ($) MAE ($) MAPE (%) Directional Acc
ARIMA (2,1,1) Multi-Step   $94.37  $85.56   10.19%          52.54%
   Holt Linear Smoothing  $164.18 $142.08   16.33%          49.15%
